# TPI - Entrega 2: Diagnostico y limpieza inicial de datos
## Comision 13 (Grupo M) | Integrante: Gustavo Farias

---

## 1. Referencia al caso y variable objetivo

Este notebook retoma el caso de reservas hoteleras (City Hotel y Resort Hotel). El problema central es comprender las cancelaciones de reservas para ayudar a la organizacion a reducir su tasa y mejorar la planificacion.

La variable objetivo es `is_canceled`:
- Valor 0: la reserva fue efectiva.
- Valor 1: la reserva fue cancelada.

En esta etapa no se modifican datos automaticamente: primero se **diagnostica**, luego se **justifica** y recien despues se aplica un tratamiento, si corresponde.

## 2. Importacion de librerias y carga del dataset

In [ ]:
## 2. Importacion de librerias
import pandas as pd
import numpy as np

# Carga del dataset
try:
    df_original = pd.read_csv('hotel booking TPI grupo M.csv')
    print("Dataset original cargado OK:", df_original.shape)
except FileNotFoundError:
    print("Error: no se encontro el archivo.")

Dataset original cargado OK: (25000, 32)


In [ ]:
# Copia de trabajo: el original queda intacto para comparaciones futuras
df = df_original.copy()
print("Copia de trabajo creada:", df.shape)

Copia de trabajo creada: (25000, 32)


## 3. Diagnostico y limpieza inicial de datos

### 3.1 Estructura, dimensiones y tipos de datos

In [ ]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print("\nNombres de las variables:")
print(list(df.columns))
print()
df.info()

Filas: 25000
Columnas: 32

Nombres de las variables:
['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  object 
 1   hotel                           25000 non-null  o

**Observaciones:**
- El dataset mantiene 25.000 filas y 32 columnas, igual que en la Entrega 1.
- `children`, `agent` y `company` aparecen como `float64` aunque representan conteos o identificadores. Se podrian transformar despues de tratar los nulos.
- Las variables temporales (`arrival_date`, `arrival_date_month`) siguen como texto (`object`).

### 3.2 Valores faltantes y cadenas vacias

In [ ]:
# Valores faltantes
nulos = df.isnull().sum()
pct = (df.isnull().mean() * 100).round(2)
tabla_nulos = pd.DataFrame({'nulos': nulos, 'porcentaje': pct})
print(tabla_nulos[tabla_nulos['nulos'] > 0].sort_values('nulos', ascending=False))

# Strings vacios o solo espacios en columnas de texto
print()
obj = df.select_dtypes(include='object')
cadenas_vacias = obj.apply(lambda c: c.str.strip().eq('').sum())
print("Cadenas vacias por columna:")
print(cadenas_vacias[cadenas_vacias > 0])

          nulos  porcentaje
company   23586       94.34
agent      3447       13.79
country     104        0.42
children      1        0.00

Cadenas vacias por columna:
Series([], dtype: int64)


**Observaciones:**
- Los faltantes se concentran en `company` (94,34%), `agent` (13,79%), `country` (0,42%) y `children` (1 caso).
- No se detectan string vacios: los faltantes ya vienen codificados como null.
- No se analizan ahora faltantes.

### 3.3 Categorias inconsistentes

In [ ]:
for col in ['hotel', 'meal', 'market_segment', 'distribution_channel', 'deposit_type', 'customer_type']:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

print("--- country: codigos distintos ---")
print("Cantidad de codigos:", df['country'].nunique())
print(sorted(df['country'].dropna().unique()))

--- hotel ---
hotel
City Hotel      16522
Resort Hotel     8478
Name: count, dtype: int64

--- meal ---
meal
BB           19404
HB            2976
SC            2220
Undefined      232
FB             168
Name: count, dtype: int64

--- market_segment ---
market_segment
Online TA        11811
Offline TA/TO     4992
Groups            4181
Direct            2735
Corporate         1090
Complementary      138
Aviation            52
Undefined            1
Name: count, dtype: int64

--- distribution_channel ---
distribution_channel
TA/TO        20417
Direct        3158
Corporate     1380
GDS             44
Undefined        1
Name: count, dtype: int64

--- deposit_type ---
deposit_type
No Deposit    21912
Non Refund     3051
Refundable       37
Name: count, dtype: int64

--- customer_type ---
customer_type
Transient          18807
Transient-Party     5224
Contract             847
Group                122
Name: count, dtype: int64

--- country: codigos distintos ---
Cantidad de codigos: 123
['AG

**Observaciones:**
- En `meal` aparece la categoria `Undefined`, que seria un faltante encubierto.
- En `country` parece que los codigos `CN` y `CHN` significan el mismo pais (China). Si es asi, tendria que unificarlos mas adelante.
- El resto de las categorias (hotel, market_segment, distribution_channel, deposit_type, customer_type) se ven consistentes.

### 3.4 Registros duplicados y reserva unica

In [ ]:
print("Filas 100% duplicadas:", df.duplicated().sum())
print("booking_id duplicados:", df['booking_id'].duplicated().sum())
print("booking_id unicos:", df['booking_id'].nunique())

Filas 100% duplicadas: 0
booking_id duplicados: 0
booking_id unicos: 25000


**Observaciones:**
- `booking_id` no presenta duplicados, por lo que funciona como **identificador unico de cada reserva**. Esto confirma que no hay reservas cargadas dos veces y que cualquier analisis puede apoyarse en este campo como clave.

### 3.5 Valores imposibles, poco esperables o atipicos

#### Huespedes: children, adults y babies

In [ ]:
# children: faltantes, tipo de dato y valores
print("Nulos en children:", df['children'].isnull().sum())
print()
print("Tipo de dato:", df['children'].dtype)
print()
print(df['children'].value_counts())
print()

# adults: valores imposibles
print("Reservas con mas de 10 adultos:")
print(df[df['adults'] > 10][['booking_id', 'adults', 'children', 'babies', 'is_canceled']])
print()

# Reservas sin huespedes
sin_huespedes = (df['adults'] == 0) & (df['children'].fillna(0) == 0) & (df['babies'] == 0)
print("Reservas sin huespedes:", sin_huespedes.sum())

Nulos en children: 1

Tipo de dato: float64

children
0.0    23196
1.0     1021
2.0      768
3.0       14
Name: count, dtype: int64

Reservas con mas de 10 adultos:
      booking_id  adults  children  babies  is_canceled
20172  HB-001644      50       0.0       0            1

Reservas sin huespedes: 42


**Observaciones:**
- `children` tiene 1 faltante y viene como float: habra que agregarlo y luego convertirlo a entero.
- Aparecen reservas con `adults` = 50, una cantidad imposible para una reserva comun: habria que revisar ese registro puntual.
- Existen reservas sin huespedes (adults=0, children=0, babies=0), lo cual tambien habria revisar mas adelante.

#### Duracion de la estadia

In [ ]:
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

print("Estadias de 0 noches:", (df['total_nights'] == 0).sum())
print()
print("Distribucion por is_canceled en estadias de 0 noches:")
print()
print(df.loc[df['total_nights'] == 0, 'is_canceled'].value_counts())

Estadias de 0 noches: 150

Distribucion por is_canceled en estadias de 0 noches:

is_canceled
0    144
1      6
Name: count, dtype: int64


**Observaciones:**
- Hay estadias de 0 noches. En reservas canceladas podria ser porque la estadia nunca se concreto.
- Tambien aparecen algunas reservas **no canceladas** con 0 noches: son casos a revisar tambien.

#### Tarifa (adr) y anticipacion (lead_time)

In [ ]:
# adr: valores cero o negativos
print("adr menor o igual a 0:", (df['adr'] <= 0).sum())
print()
print(df[df['adr'] == 0][['booking_id', 'adr', 'meal', 'market_segment', 'customer_type', 'is_canceled']].head(10))
print()

# lead_time: anticipaciones poco habituales
print("lead_time mayor a 365 dias:", (df['lead_time'] > 365).sum())
print("lead_time maximo:", df['lead_time'].max())

adr menor o igual a 0: 395

    booking_id  adr meal market_segment    customer_type  is_canceled
40   HB-089113  0.0   HB  Offline TA/TO        Transient            0
107  HB-056402  0.0   BB  Complementary        Transient            1
126  HB-005992  0.0   BB  Complementary        Transient            0
132  HB-078210  0.0   BB  Complementary        Transient            1
210  HB-083179  0.0   BB  Complementary        Transient            0
290  HB-087100  0.0   HB         Groups  Transient-Party            0
292  HB-112098  0.0   BB  Complementary        Transient            0
429  HB-011543  0.0   SC      Corporate        Transient            0
454  HB-107155  0.0   BB  Offline TA/TO        Transient            0
456  HB-010434  0.0   BB  Complementary        Transient            1

lead_time mayor a 365 dias: 646
lead_time maximo: 737


**Observaciones:**
- Varios casos de `adr` = 0 y ocurren en segmentos como Complementary o reservas de grupo. Podria ser una atencion a ciertos clientes o un error.Habria que revisarlo con mas detalle.
- `lead_time` supera el año de anticipacion en varios casos (maximo 737 dias). Otro punto a revisar mas adelante.

#### Variables temporales: formato, validez y coherencia

In [ ]:
df['fecha_parseada'] = pd.to_datetime(df['arrival_date'], errors='coerce')

print("Fechas invalidas:", df['fecha_parseada'].isnull().sum())
print("Año no coincide con arrival_date_year:", (df['fecha_parseada'].dt.year != df['arrival_date_year']).sum())
print("Dia no coincide con arrival_date_day_of_month:", (df['fecha_parseada'].dt.day != df['arrival_date_day_of_month']).sum())

Fechas invalidas: 0
Año no coincide con arrival_date_year: 0
Dia no coincide con arrival_date_day_of_month: 0


**Observaciones:**
- No detecto fechas invalidas ni inconsistencias entre `arrival_date` y sus componentes (año y dia).
- De todos modos, `arrival_date` es un texto, por lo que se podria parsear a datetime. En todo caso, se transformaria mas adelante a modo de limpieza.

## 4. Bitacora de decisiones

| Problema detectado | Variable | Decision | Justificacion |
|---|---|---|---|
| 1 valor faltante | `children` | Corregir con 0 | Es un solo caso (0,004%). La mayoria de las reservas no tiene children, por lo que 0 es un valor razonable. Necesario antes de convertir a entero. |
| 104 faltantes (0,42%) | `country` | Revisar | Porcentaje minimo. Se puede conservar como "Sin dato" o excluir solo en analisis por pais. No afecta el diagnostico general. |
| 3.447 faltantes (13,79%) | `agent` | Conservar | El faltante tiene sentido de negocio: reserva sin agente de viajes y modificarlo podria distorsionarlo. |
| 23.586 faltantes (94,34%) | `company` | Conservar | Estructural: casi todos los huespedes son particulares. Aporta poco, pero no lo eliminaria sin documentarlo. |
| Tipo float64 en conteos | `children`, `agent`, `company` | Transformar | Son conteos o identificadores. Convertir a int/categoria despues de tratar faltantes. |
| Codigos `CN` y `CHN` | `country` | Unificar | Dos codigos para el mismo pais fragmentan las frecuencias y sesgan el analisis por pais. |
| Categoria `Undefined` | `meal` | Revisar | Funciona como faltante encubierto. Documentar y revisar si corresponde renombrar a "Sin dato". |
| `adr` igual a 0 | `adr` | Conservar y documentar | Se concentra en segmentos Complementary: sugiere cortesia, no error. No se elimina sin justificar. |
| `adults` = 50 | `adults` | Revisar/Corregir | Cantidad imposible para una reserva comun. Revisar el registro puntual antes de corregir. |
| Reservas sin huespedes | `adults`, `children`, `babies` | Revisar | Combinacion poco rara. Verificar si corresponde a cancelaciones o a errores de carga. |
| Estadias de 0 noches | `stays_in_weekend_nights`, `stays_in_week_nights` | Conservar y documentar | Esperable en canceladas; revisar las no canceladas con 0 noches antes de analizar duracion o ingresos. |
| `lead_time` > 365 dias | `lead_time` | Conservar | Poco habitual pero posible. No se elimina solo por ser extremo. |
| Fechas como texto | `arrival_date` | Transformar | Parsear a datetime para habilitar analisis temporal. Verifique que no hay fechas invalidas. |
| Identificador de reserva | `booking_id` | Conservar como clave | Sin duplicados: cada fila es una reserva unica. Base para validaciones futuras. |

---

## 5. Conclusion preliminar y proximos pasos

El dataset es utilizable para el objetivo de analizar cancelaciones, pero requiere tratamientos puntuales: modificar `children`, unificar codigos de pais, documentar `Undefined` en `meal`, convertir tipos y revisar casos imposibles (adults = 50, reservas sin huespedes, estadias de 0 noches no canceladas).

---

# PARTE 2: TRANSFORMACION Y PREPARACION (SEMANA 5)

A continuacion, se aplican los tratamientos justificados en la bitacora de la Semana 4 y se crean las variables derivadas necesarias para el analisis, cumpliendo con los requisitos de la Entrega 2.

## 6. Transformacion de variables y limpieza aplicada
Basado en el diagnostico de la Semana 4, aplicamos los tratamientos justificados en la bitacora sobre la copia de trabajo `df`.

### 6.1 Tratamiento de valores faltantes y tipos de datos

In [ ]:
# 1. children: imputamos el unico faltante con 0 y convertimos a entero
df['children'] = df['children'].fillna(0).astype(int)

# 2. agent y company: conservo los faltantes pero convertimos a categoria/entero cuando aplique
# Para agent, convertimos a Int64 (entero nullable) para no perder los NaN como informacion
df['agent'] = df['agent'].astype('Int64')

# Para company, como es 94% nulo y aporta poco, no se elimina la columna y se documenta su baja utilidad.

print("Tipos de datos despues de correccion:")
print()
print(df[['children', 'agent', 'company']].dtypes)
print()
print("Nulos restantes en children:", df['children'].isnull().sum())

Tipos de datos despues de correccion:

children      int64
agent         Int64
company     float64
dtype: object

Nulos restantes en children: 0


**Justificacion:**
- `children` tenia 1 faltante (0,004%). Imputar con 0 es razonable porque la mayoria de las reservas no tiene niños. Se convierte a `int` porque es un conteo.
- `agent` se convierte a `Int64` (entero que acepta nulos) para mantener la distincion entre "sin agente" (NaN) y un ID de agente real.
- `company` se conserva tal cual; su alta tasa de nulos (94%) refleja que la mayoria son clientes particulares.

### 6.2 Unificacion de categorias inconsistentes

In [ ]:
# Unificar codigos de pais: CN y CHN -> CHN
print("Antes:", df['country'].value_counts().loc[['CHN', 'CN']].to_dict())

df['country'] = df['country'].replace('CN', 'CHN')

print("Despues:", df['country'].value_counts().loc[['CHN']].to_dict())
print()

# Tratar 'Undefined' en meal como categoria explicita o dejarlo
# Lo dejamos como 'Undefined' pero lo documentamos como "Sin dato / No especificado"
print("Categorias en meal:")
print(df['meal'].value_counts())

Antes: {'CHN': 231, 'CN': 264}
Despues: {'CHN': 495}

Categorias en meal:
meal
BB           19404
HB            2976
SC            2220
Undefined      232
FB             168
Name: count, dtype: int64


**Justificacion:**
- Se unifican `CN` y `CHN` bajo `CHN` para evitar fragmentar el analisis por pais.
- `Undefined` en `meal` se conserva como categoria valida que representa "tipo de comida no especificado", evitando eliminar registros valiosos.

### 6.3 Transformacion de variables temporales

In [ ]:
# Convertir arrival_date a datetime
df['arrival_date'] = pd.to_datetime(df['arrival_date'], errors='coerce')

# Verificar que no haya fechas invalidas
print("Fechas invalidas despues de parseo:", df['arrival_date'].isnull().sum())
print("Tipo de dato:", df['arrival_date'].dtype)

Fechas invalidas despues de parseo: 0
Tipo de dato: datetime64[ns]


**Justificacion:**
- Se parsea `arrival_date` a `datetime64` para habilitar analisis temporales (estacionalidad, dias de la semana).
- Ya habia verificado en la Semana 4 que no haya inconsistencias entre la fecha y sus compontentes.

## 7. Variables derivadas

Creo nuevas columnas que me sean utiles para responder las preguntas de analisis, basadas en las combinaciones de variables existentes.

In [ ]:
# 1. Duracion total de la estadia
# total_nights ya existe en df desde la celda de diagnostico, pero la recalculo por seguridad
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

# 2. Cantidad total de huespedes
df['total_guests'] = df['adults'] + df['children'] + df['babies']

# 3. Importe estimado de la reserva (adr * total_nights)
# Solo tiene sentido para reservas con estadia > 0. Para 0 noches, el importe es 0.
df['estimated_revenue'] = df['adr'] * df['total_nights']

# 4. Reserva familiar (heuristica: si tiene children o bebes > 0)
df['is_family_booking'] = ((df['children'] > 0) | (df['babies'] > 0)).astype(int)

# 5. Segmentos de anticipacion (discretizacion de lead_time)
def segmentar_lead_time(dias):
    if dias <= 7:
        return 'Ultima hora (0-7 dias)'
    elif dias <= 30:
        return 'Corto plazo (8-30 dias)'
    elif dias <= 90:
        return 'Mediano plazo (31-90 dias)'
    else:
        return 'Largo plazo (>90 dias)'

df['lead_time_segment'] = df['lead_time'].apply(segmentar_lead_time)

# 6. Dia de la semana de llegada (0=Lunes, 6=Domingo)
df['arrival_day_of_week'] = df['arrival_date'].dt.day_name()

# Muestra las primeras filas de las nuevas variables
print("Nuevas variables creadas:")
print()
print(df[['booking_id', 'total_nights', 'total_guests', 'estimated_revenue', 'is_family_booking', 'lead_time_segment', 'arrival_day_of_week']].head(10))

# Elimino la columna temporal de diagnostico para dejar el dataset limpio
df.drop(columns=['fecha_parseada'], inplace=True)

print("\nDataset final preparado:", df.shape)

Nuevas variables creadas:

  booking_id  total_nights  total_guests  estimated_revenue  \
0  HB-000924             3             2             361.80   
1  HB-004486             3             2             180.00   
2  HB-112184             3             2             570.00   
3  HB-019323             4             2             171.00   
4  HB-005173             7             2             485.52   
5  HB-113423             2             3             340.00   
6  HB-083793             1             2               2.00   
7  HB-003291             2             2              82.80   
8  HB-034278            21             1             767.55   
9  HB-053332             4             3             708.00   

   is_family_booking           lead_time_segment arrival_day_of_week  
0                  0      Largo plazo (>90 dias)              Sunday  
1                  0     Corto plazo (8-30 dias)           Wednesday  
2                  0     Corto plazo (8-30 dias)              Sund

**Justificacion de las variables derivadas:**
- `total_nights`: Permite analizar la duracion real de la estadia y filtrar casos de 0 noches.
- `total_guests`: Simplifica el analisis de ocupacion y capacidad de habitaciones.
- `estimated_revenue`: Aproxima el ingreso potencial de la reserva (util para analizar impacto economico de cancelaciones).
- `is_family_booking`: Permite segmentar entre clientes familiares y no familiares para ver si hay diferencias en tasas de cancelacion.
- `lead_time_segment`: Discretiza la anticipacion en categorias interpretables (ultima hora, corto, mediano, largo plazo) para facilitar comparaciones.
- `arrival_day_of_week`: Permite analizar estacionalidad semanal (por ejemplo, si se cancela mas los fines de semana).

## 8. Bitacora del proceso (Actualizada)

Agregamos los nuevos registros correspondientes a las transformaciones y variables derivadas aplicadas.

| Problema detectado / Necesidad | Variable | Decision | Justificacion |
|---|---|---|---|
| 1 valor faltante en conteo | `children` | Imputar con 0 y convertir a int | Valor razonable (mayoria sin niños). Necesario para calculos de huespedes totales. |
| Tipos float en identificadores | `agent` | Convertir a Int64 | Mantiene los nulos como informacion ("sin agente") pero usa tipo numerico correcto. |
| Codigos duplicados de pais | `country` | Reemplazar CN por CHN | Estandariza los nombres y evita fragmentacion en analisis por pais. |
| Fecha como texto | `arrival_date` | Parsear a datetime | Habilita extraccion de dia de la semana y analisis temporal. |
| Necesidad de duracion total | `total_nights` | Crear variable derivada | Suma de noches de fin de semana y semana. Base para calcular ingresos. |
| Necesidad de ocupacion total | `total_guests` | Crear variable derivada | Suma de adultos, niños y bebes. Simplifica analisis de capacidad. |
| Necesidad de impacto economico | `estimated_revenue` | Crear variable derivada | adr * total_nights. Permite cuantificar perdidas por cancelacion. |
| Segmentacion de clientes | `is_family_booking` | Crear variable binaria | Identifica reservas con niños/bebes para analisis comparativo. |
| Anticipacion dificil de interpretar | `lead_time_segment` | Discretizar en 4 categorias | Agrupa dias en rangos interpretables (ultima hora, corto, mediano, largo plazo). |
| Analisis de estacionalidad semanal | `arrival_day_of_week` | Extraer nombre del dia | Permite comparar cancelaciones entre lunes-viernes y fines de semana. |

## 9. Dataset preparado: Estado final

El conjunto de datos ha sido diagnosticado, limpiado y enriquecido con variables derivadas sobre la copia de trabajo `df`, conservando el `df_original` intacto.

**Resumen del estado final:**
- **Filas:** 25.000 (no se eliminaron registros durante la limpieza, respetando la trazabilidad y el volumen de datos).
- **Columnas originales:** 32.
- **Columnas nuevas:** 6 (`total_nights`, `total_guests`, `estimated_revenue`, `is_family_booking`, `lead_time_segment`, `arrival_day_of_week`).
- **Total de columnas:** 38.

**Preparacion para analisis posteriores (Unidad 3):**
El dataset `df` esta optimizado para responder las preguntas iniciales del negocio:
1. **Calidad asegurada:** Los faltantes criticos (`children`) fueron imputados y convertidos a entero; los tipos de datos son correctos (`Int64` para agentes, `datetime64` para fechas) y las categorias estan unificadas (codigos `CN` y `CHN` consolidados).
2. **Variables listas para analizar:**
   - *Anticipacion vs Cancelacion:* Se usara `lead_time_segment` cruzado con `is_canceled`.
   - *Impacto economico:* Se usara `estimated_revenue` para cuantificar las perdidas por cancelaciones.
   - *Perfil del cliente:* Se usara `is_family_booking` y `total_guests` para ver si las familias cancelan mas o menos que otros perfiles.
   - *Estacionalidad:* Se usara `arrival_day_of_week` para detectar patrones semanales.
3. **Trazabilidad:** Cada decision (como conservar `adr = 0` o los nulos de `company`) esta documentada en la Bitacora de decisiones, garantizando que el proceso sea reproducible, etico y auditable.